<a href="https://colab.research.google.com/github/ahmadtza/ahmadtza/blob/main/motor_fault_detection_using_emd_and_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CONFIGS**

In [1]:
FS = 1000
WINDOW_LEN = 800
WINDOW_STEP = 400
MAX_IMFS = 10
N_FOLDS = 10
!pip install EMD-signal

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.2/82.2 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.0/120.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.3/150.3 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 2.2 MB/s eta 0:00:00
  Attempting uninstall: dill
    Found existing installation: dill 0.3.8
    Uninstalling dill-0.3.8:
      Successfully uninstalled dill-0.3.8
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.16
    Uninstalling multiprocess-0.70.16:
      Successfully uninstalled multiprocess-0.70.16
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
datasets 4.0.0 requires dill<0.3.9,>=0.3.0, but you have dill 0.4.1 which is incompatible.
datasets 4.0.0

**DATA LOADING**

In [2]:
from pathlib import Path

from scipy.io import loadmat
DATA_ROOT = "/kaggle/input/datasets/sumairaziz/vibration-faults-dataset-for-rotating-machines"

def load_data_custom(file_name: str, directory: str, data_root: Path = DATA_ROOT):
    return loadmat(data_root / directory / file_name)


def section_dataset(data_root: Path = DATA_ROOT):

    data_root = Path(data_root)

    healthy = {"fileName": [], "x": [], "label": 0, "predicted": []}
    faulty = {"fileName": [], "x": [], "label": 1, "predicted": []}

    healthy_dir = data_root / "Healthy"
    faulty_dir = data_root / "Faulty"

    for item in sorted(healthy_dir.iterdir()):
        if item.is_file():
            healthy["fileName"].append(item.name)
            healthy["x"].append(load_data_custom(item.name, "Healthy", data_root)["H"].squeeze())

    for item in sorted(faulty_dir.iterdir()):
        if item.is_file():
            faulty["fileName"].append(item.name)
            faulty["x"].append(load_data_custom(item.name, "Faulty", data_root)["H"].squeeze())

    return healthy, faulty


def deterministic_file_folds(n_files: int, n_folds: int):
    folds = [[] for _ in range(n_folds)]
    for index in range(n_files):
        folds[index % n_folds].append(index)
    return folds

**FEATURE EXTRACTION**

In [3]:
import sys

import numpy as np
from PyEMD import EMD
from scipy.signal import welch
from scipy.stats import kurtosis, skew



SRC_ROOT = "/kaggle/input/notebooks/badrshehim/binary-classification-using-emd"

if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

try:
    from utils.emd_processing import sig_to_imf as user_sig_to_imf
except Exception:
    user_sig_to_imf = None


def sig_to_imf_paper(signal, max_imfs=MAX_IMFS):
    if user_sig_to_imf is not None:
        try:
            imfs_filtered, residue, reconstructed = user_sig_to_imf(signal, max_imfs=max_imfs)
            if reconstructed is not None and len(reconstructed) == len(signal):
                imfs_filtered = (
                    np.atleast_2d(imfs_filtered)
                    if getattr(imfs_filtered, "size", 0)
                    else np.array([])
                )
                residue = residue if residue is not None else np.zeros_like(signal)
                return imfs_filtered, residue, reconstructed
        except Exception:
            pass

    emd = EMD()
    emd.emd(signal)
    imfs, residue = emd.get_imfs_and_residue()
    if imfs is None or (hasattr(imfs, "size") and imfs.size == 0):
        return np.array([]), residue if residue is not None else np.zeros_like(signal), signal.copy()

    imfs = np.atleast_2d(imfs)
    if imfs.shape[0] < max_imfs:
        pad_rows = np.zeros((max_imfs - imfs.shape[0], imfs.shape[1]))
        imfs = np.vstack([imfs, pad_rows])
    else:
        imfs = imfs[:max_imfs]

    if imfs.shape[0] > 1:
        imfs_filtered = imfs[1:max_imfs]
        reconstructed = np.sum(imfs_filtered, axis=0) + (residue if residue is not None else 0.0)
    else:
        imfs_filtered = np.array([])
        reconstructed = residue if residue is not None else signal.copy()

    return imfs_filtered, residue if residue is not None else np.zeros_like(signal), reconstructed


def segment_signal(signal, window=WINDOW_LEN, step=WINDOW_STEP):
    starts = range(0, len(signal) - window + 1, step)
    return np.array([signal[start : start + window] for start in starts])


def temporal_features(signal):
    if signal.size == 0:
        return np.zeros(7)
    mean_value = np.mean(signal)
    std_value = np.std(signal)
    return np.array(
        [
            mean_value,
            std_value,
            float(skew(signal)),
            float(kurtosis(signal)),
            float(np.ptp(signal)),
            float(np.sqrt(np.mean(signal**2))),
            float(np.sum(signal**2)),
        ],
        dtype=float,
    )


def compute_psd(signal, fs=FS, nperseg=1024):
    nperseg = min(len(signal), nperseg)
    if nperseg < 8:
        return np.array([0.0]), np.array([np.sum(signal**2) + 1e-12])
    frequencies, psd = welch(signal, fs=fs, nperseg=nperseg)
    return frequencies, np.maximum(psd, 1e-12)


def freq_a_features(signal, fs=FS):
    if signal.size == 0:
        return np.zeros(6)
    frequencies, psd = compute_psd(signal, fs)
    total_energy = np.sum(psd)
    fm = np.sum(frequencies * psd) / (total_energy + 1e-12)
    fsd = np.sqrt(np.sum(((frequencies - fm) ** 2) * psd) / (total_energy + 1e-12))
    fsk = np.sum(((frequencies - fm) ** 3) * psd) / ((total_energy + 1e-12) * (fsd**3 + 1e-12))
    fkr = np.sum(((frequencies - fm) ** 4) * psd) / ((total_energy + 1e-12) * (fsd**4 + 1e-12))
    cumulative = np.cumsum(psd)
    median_index = np.searchsorted(cumulative, 0.5 * total_energy)
    fmed = float(frequencies[min(median_index, len(frequencies) - 1)])
    return np.array([fm, fsd, fsk, fkr, total_energy, fmed], dtype=float)


def freq_b_features(signal, fs=FS):
    if signal.size == 0:
        return np.zeros(8)
    frequencies, psd = compute_psd(signal, fs)
    total_energy = np.sum(psd)
    spectral_centroid = np.sum(frequencies * psd) / (total_energy + 1e-12)
    normalized = psd / (np.sum(psd) + 1e-12)
    spectral_flatness = float(np.sum((np.diff(normalized, prepend=normalized[0])) ** 2))
    cumulative = np.cumsum(psd)
    rolloff_index = np.searchsorted(cumulative, 0.85 * total_energy)
    rolloff = float(frequencies[min(rolloff_index, len(frequencies) - 1)])
    geo_mean = np.exp(np.mean(np.log(psd + 1e-12)))
    arith_mean = np.mean(psd)
    flatness_ratio = float(geo_mean / (arith_mean + 1e-12))
    crest_ratio = float(np.max(psd) / (arith_mean + 1e-12))
    if len(psd) >= 2:
        indexes = np.arange(1, len(psd) + 1)
        decay = float(np.sum((psd[:-1] - psd[1:]) / (indexes[:-1] + 1e-12)) / (np.sum(psd) + 1e-12))
    else:
        decay = 0.0
    try:
        slope, _ = np.polyfit(frequencies, psd, 1)
    except Exception:
        slope = 0.0
    spread = float(np.sqrt(np.sum(((frequencies - spectral_centroid) ** 2) * psd) / (total_energy + 1e-12)))
    return np.array(
        [spectral_centroid, spectral_flatness, rolloff, flatness_ratio, crest_ratio, decay, float(slope), spread],
        dtype=float,
    )


def compose_feature_sets_from_reconstructed(reconstructed):
    temporal = temporal_features(reconstructed)
    freq_a = freq_a_features(reconstructed)
    freq_b = freq_b_features(reconstructed)
    return {
        "F1": temporal,
        "F2": freq_a,
        "F3": freq_b,
        "F4": np.concatenate((temporal, freq_a)),
        "F5": np.concatenate((freq_a, freq_b)),
        "F6": np.concatenate((temporal, freq_b)),
        "F7": np.concatenate((temporal, freq_a, freq_b)),
    }


**MODELS**

In [4]:
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC


class MatlabSVMQ(SVC):
    def fit(self, X, y):
        variance = np.var(X, axis=0).mean() + 1e-12
        gamma_matlab = 1.0 / (X.shape[1] * variance)
        self.gamma = gamma_matlab
        self.degree = 2
        self.coef0 = 1
        self.C = 1.0
        return super().fit(X, y)


def build_models():
    return {
        "SVM-Q": MatlabSVMQ(kernel="poly"),
        "KNN-W": KNeighborsClassifier(weights="distance"),
        "LDA": LinearDiscriminantAnalysis(),
        "RF": RandomForestClassifier(random_state=0),
        "LGBM": LGBMClassifier(random_state=0, verbose=-1),
    }


**PIPELINE**

In [5]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", message="X does not have valid feature names")
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.preprocessing import MinMaxScaler, StandardScaler






ARTIFACTS_DIR = Path("/kaggle/working/artifacts")
FIGURES_DIR = Path("/kaggle/working/figures")
CONFUSION_DIR = FIGURES_DIR / "confusion_matrices"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


def build_window_dataset(healthy, faulty):
    files = []

    def process_file_signal(array, label, file_name, global_index):
        x_axis = array[:, 0]
        y_axis = array[:, 1]
        z_axis = array[:, 2]
        magnitude = np.sqrt(x_axis**2 + y_axis**2 + z_axis**2)
        magnitude = magnitude / (np.max(np.abs(magnitude)) + 1e-12)
        segments = segment_signal(magnitude)
        window_features = []
        for segment in segments:
            _, _, reconstructed = sig_to_imf_paper(segment)
            window_features.append(compose_feature_sets_from_reconstructed(reconstructed))
        return {
            "file_name": file_name,
            "label": label,
            "window_features": window_features,
            "n_windows": len(window_features),
            "global_index": global_index,
        }

    global_index = 0
    for index, file_name in enumerate(healthy["fileName"]):
        files.append(process_file_signal(healthy["x"][index], 0, file_name, global_index))
        global_index += 1

    for index, file_name in enumerate(faulty["fileName"]):
        files.append(process_file_signal(faulty["x"][index], 1, file_name, global_index))
        global_index += 1

    return files


def run_paper_cv(healthy, faulty):
    ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
    CONFUSION_DIR.mkdir(parents=True, exist_ok=True)

    files = build_window_dataset(healthy, faulty)
    normal_count = len(healthy["fileName"])
    faulty_count = len(faulty["fileName"])

    normal_folds = deterministic_file_folds(normal_count, N_FOLDS)
    faulty_folds = deterministic_file_folds(faulty_count, N_FOLDS)
    normal_global = list(range(0, normal_count))
    faulty_global = list(range(normal_count, normal_count + faulty_count))
    normal_folds_global = [[normal_global[i] for i in fold] for fold in normal_folds]
    faulty_folds_global = [[faulty_global[i] for i in fold] for fold in faulty_folds]

    records = []
    models = build_models()

    for feature_set in ["F1", "F2", "F3", "F4", "F5", "F6", "F7"]:
        for model_name, base_model in models.items():
            fold_accs = []
            fold_precs = []
            fold_recs = []
            fold_f1s = []
            total_tp = total_tn = total_fp = total_fn = 0

            for fold_index in range(N_FOLDS):
                test_file_indices = normal_folds_global[fold_index] + faulty_folds_global[fold_index]
                train_file_indices = [index for index in range(len(files)) if index not in test_file_indices]
                train_vectors = []
                train_labels = []
                test_file_windows = {}
                test_file_labels = {}

                for file_index in train_file_indices:
                    file_object = files[file_index]
                    for window_sets in file_object["window_features"]:
                        train_vectors.append(window_sets[feature_set])
                        train_labels.append(file_object["label"])

                for file_index in test_file_indices:
                    file_object = files[file_index]
                    window_vectors = [window_sets[feature_set] for window_sets in file_object["window_features"]]
                    test_file_windows[file_index] = np.vstack(window_vectors) if window_vectors else np.zeros((0, 1))
                    test_file_labels[file_index] = file_object["label"]

                X_train = np.vstack(train_vectors) if train_vectors else np.zeros((0, 1))
                y_train = np.array(train_labels, dtype=int)
                scaler = MinMaxScaler() if model_name == "SVM-Q" else StandardScaler()
                X_train_scaled = scaler.fit_transform(X_train)

                model = clone(base_model)

                model.fit(X_train_scaled, y_train
                         )

                y_true_files = []
                y_pred_files = []
                for file_index in test_file_indices:
                    X_test_windows = test_file_windows[file_index]
                    if X_test_windows.size == 0:
                        continue
                    y_window = model.predict(scaler.transform(X_test_windows))
                    counts = np.bincount(y_window.astype(int))
                    if len(counts) == 0:
                        prediction = 0
                    elif len(counts) == 1:
                        prediction = int(np.argmax(counts))
                    elif counts[0] == counts[1]:
                        prediction = 1
                    else:
                        prediction = int(np.argmax(counts))
                    y_true_files.append(test_file_labels[file_index])
                    y_pred_files.append(prediction)

                y_true_files = np.array(y_true_files, dtype=int)
                y_pred_files = np.array(y_pred_files, dtype=int)
                acc = accuracy_score(y_true_files, y_pred_files)
                prec = precision_score(y_true_files, y_pred_files, zero_division=0)
                rec = recall_score(y_true_files, y_pred_files, zero_division=0)
                f1 = f1_score(y_true_files, y_pred_files, zero_division=0)
                fold_accs.append(acc)
                fold_precs.append(prec)
                fold_recs.append(rec)
                fold_f1s.append(f1)

                cm = confusion_matrix(y_true_files, y_pred_files)
                if cm.size == 4:
                    tn, fp, fn, tp = cm.ravel()
                else:
                    tn = cm[0, 0] if cm.shape == (1, 1) else 0
                    fp = fn = tp = 0

                total_tp += int(tp)
                total_fp += int(fp)
                total_tn += int(tn)
                total_fn += int(fn)

                fold_df = pd.DataFrame(
                    {
                        "FeatureSet": [feature_set],
                        "Model": [model_name],
                        "Fold": [fold_index + 1],
                        "Accuracy": [acc],
                        "Precision": [prec],
                        "Recall": [rec],
                        "F1": [f1],
                        "TP": [int(tp)],
                        "TN": [int(tn)],
                        "FP": [int(fp)],
                        "FN": [int(fn)],
                    }
                )
                fold_df.to_csv(ARTIFACTS_DIR / f"{feature_set}_{model_name}_fold{fold_index + 1}.csv", index=False)

                disp = ConfusionMatrixDisplay(cm, display_labels=["Normal", "Faulty"])
                disp.plot()
                plt.title(f"{feature_set} - {model_name} - Fold {fold_index + 1}")
                plt.savefig(CONFUSION_DIR / f"{feature_set}_{model_name}_cm_fold{fold_index + 1}.png")
                plt.close()

            record = {
                "FeatureSet": feature_set,
                "Model": model_name,
                "AvgAccuracy": float(np.mean(fold_accs)),
                "StdAccuracy": float(np.std(fold_accs)),
                "AvgPrecision": float(np.mean(fold_precs)),
                "AvgRecall": float(np.mean(fold_recs)),
                "AvgF1": float(np.mean(fold_f1s)),
                "TotalTP": int(total_tp),
                "TotalTN": int(total_tn),
                "TotalFP": int(total_fp),
                "TotalFN": int(total_fn),
                "NumFolds": N_FOLDS,
                "Samples": len(files),
            }
            records.append(record)
            print(f"Done: {feature_set} - {model_name} | F1={record['AvgF1']:.4f} Acc={record['AvgAccuracy']:.4f}")

    dataframe = pd.DataFrame(records).sort_values(by=["AvgF1", "AvgAccuracy"], ascending=False).reset_index(drop=True)
    dataframe.to_csv(ARTIFACTS_DIR / "full_model_feature_comparison.csv", index=False)
    joblib.dump(dataframe, ARTIFACTS_DIR / "full_model_feature_comparison.joblib")
    return dataframe


def main(data_root: Path = DATA_ROOT):
    healthy, faulty = section_dataset(data_root)
    print(f"Loaded {len(healthy['x'])} healthy files and {len(faulty['x'])} faulty files.")
    ranking = run_paper_cv(healthy, faulty)
    print(f"Saved final ranking CSV to: {ARTIFACTS_DIR / 'full_model_feature_comparison.csv'}")
    print(ranking.head(10))




In [ ]:
from pathlib import Path

if __name__ == "__main__":
    # Replace '/content/data' with the actual path to your dataset in Colab
    # For example, if you mount your Google Drive: data_root = Path('/content/drive/MyDrive/your_dataset_folder')
    # If you upload data to Colab's default /content directory: data_root = Path('/content/your_dataset_folder')
    # If you are using a mounted Kaggle dataset, ensure it's mounted correctly and update the path accordingly.

    # Currently, the original DATA_ROOT is defined as:
    # DATA_ROOT = '/kaggle/input/datasets/sumairaziz/vibration-faults-dataset-for-rotating-machines'
    # Since this path does not exist, we'll try to provide a placeholder for a new root.
    new_data_root = Path('/content/data') # Placeholder path, update as needed

    # You need to ensure the dataset is available at new_data_root.
    # If you want to use the original DATA_ROOT and have mounted the Kaggle dataset,
    # you can call main() without arguments or pass DATA_ROOT directly.

    main(data_root=new_data_root)

